# ba_shapes_structure_removal_experiment_fixed
BA-Shapes structure-removal study with step-by-step plot generation.

## Setup


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from notebook_runner import run_notebook_script


## Configuration


In [ ]:
SCRIPT_NAME = '05_ba_shapes_structure_removal'
overrides = {
    'render_all_plots': False,  # plot figure-by-figure below
}


## Compute Shared State


In [ ]:
state = run_notebook_script(SCRIPT_NAME, overrides=overrides)
print('Plot dir:', state['PLOT_DIR'])
print('Graph N/E:', state['N'], state['E'])

print('Palette:', {'eventblue': state['EVENT_BLUE'], 'snapshotorange': state['SNAPSHOT_ORANGE'], 'edgegray': state['EDGE_GRAY']})

### Plot 1: Example House Motif + BA Attachment


In [ ]:
state['draw_subgraph'](
    state['G'],
    state['nodes_to_plot'],
    state['y'],
    title='One house motif + its BA attachment',
    seed=state['SEED'],
    save_as='house_attachment_subgraph',
)


### Plot 2: Accuracy Curves


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(state['history']['epoch'], state['history']['train_acc'], label='train_acc', color=state['EVENT_BLUE'])
plt.plot(state['history']['epoch'], state['history']['val_acc'], label='val_acc', color=state['SNAPSHOT_ORANGE'])
plt.plot(state['history']['epoch'], state['history']['test_acc'], label='test_acc', color=state['EDGE_GRAY'])
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.title('Accuracy curves')
plt.legend()
state['savefig_pdf']('accuracy_curves')
plt.show()


### Plot 3: Training Loss Curve


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(state['history']['epoch'], state['history']['loss'], color=state['EVENT_BLUE'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss')
state['savefig_pdf']('loss_curve')
plt.show()


### Plot 4: Variant Summary Table


In [ ]:
df = state['df'].copy()
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string(index=False))


### Plot 5: Accuracy After Edge Removals


In [ ]:
df = state['df']
plt.figure(figsize=(7,4))
plt.bar(df['variant'], df['test_acc'], color=state['EVENT_BLUE'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('test accuracy (no retrain)')
plt.title('Accuracy after edge removals')
plt.tight_layout()
state['savefig_pdf']('accuracy_after_edge_removals')
plt.show()


### Plot 6: Prediction Flips vs Original


In [ ]:
df = state['df']
plt.figure(figsize=(7,4))
plt.bar(df['variant'], df['flip_rate_test'], color=state['SNAPSHOT_ORANGE'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('fraction of test nodes that flip')
plt.title('Prediction flips vs original')
plt.tight_layout()
state['savefig_pdf']('prediction_flips_vs_original')
plt.show()


### Plot 7: Per-Node Probability Shift (Original vs No Motif Edges)


In [ ]:
p_o = np.asarray(state['p_o'])
p_m = np.asarray(state['p_m'])
node_idx = int(state['node_idx'])

plt.figure(figsize=(7,4))
x = np.arange(len(p_o))
plt.bar(x - 0.2, p_o, width=0.4, label='original', color=state['EVENT_BLUE'])
plt.bar(x + 0.2, p_m, width=0.4, label='no motif edges', color=state['SNAPSHOT_ORANGE'])
plt.xticks(x, [str(i) for i in range(len(p_o))])
plt.ylabel('probability')
plt.title(f'Node {node_idx}: probability shift after removing motif edges')
plt.legend()
state['savefig_pdf'](f'node_{node_idx}_prob_shift_no_motif')
plt.show()
